# CS781 Phase 3 — Rhetorical Role Prediction (LoRA)

LoRA fine-tunes the Phase 1 Legal Language Model for sentence-level
rhetorical-role classification (13 labels).

**Why LoRA:** Full fine-tuning overfits with only 16k training sentences.
LoRA trains ~0.5M params (vs 72M) — regularizes naturally, fits in less
VRAM, trains faster, generalizes better.

**Setup:** Upload `best_enc.pt`, `bpe_40k.json`, `a1_tokenizer.py` to
Google Drive. Phase 3 data is downloaded from Hugging Face.

**Compute:** T4 (14GB) or L4 (22GB).

In [ ]:
%pip install -U torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 91.4 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [ ]:
# CELL 1 — LOAD PHASE 1 ASSETS FROM GOOGLE DRIVE AND PHASE 3 DATA FROM HUGGING FACE

from pathlib import Path
from google.colab import drive
from getpass import getpass
from huggingface_hub import HfApi, hf_hub_download
import os, re, sys, subprocess

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub", "peft"])

drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive")

print("Searching Google Drive for Phase 1 assets...")
drive_files = []
for root, dirs, files in os.walk(DRIVE_ROOT):
    dirs[:] = [d for d in dirs if d not in {".Trash", ".ipynb_checkpoints", "node_modules"}]
    for filename in files:
        drive_files.append(Path(root) / filename)

def normalized_name(path):
    return re.sub(r" \(\d+\)(?=\.)", "", path.name).lower()

def locate_drive_file(filename, preferred_folder=None):
    target = filename.lower()
    matches = [p for p in drive_files if p.name.lower() == target or normalized_name(p) == target]
    if preferred_folder:
        preferred = [p for p in matches if preferred_folder.lower() in str(p.parent).lower()]
        if preferred:
            matches = preferred
    if not matches:
        raise FileNotFoundError(f"Could not find {filename} on Google Drive.")
    matches.sort(key=lambda p: (p.name.lower() != target, len(p.parts), str(p).lower()))
    if len(matches) > 1:
        print(f"Multiple matches for {filename}; using {matches[0]}")
    return matches[0]

CHECKPOINT_PATH = locate_drive_file("best_enc.pt", "Phase_1")
TOKENIZER_JSON = locate_drive_file("bpe_40k.json")
TOKENIZER_PY = locate_drive_file("a1_tokenizer.py")

HF_REPO = "Exploration-Lab/CS-781-Capstone"
HF_TOKEN = getpass("Hugging Face read token (input hidden): ")

api = HfApi()
repo_files = api.list_repo_files(repo_id=HF_REPO, repo_type="dataset", token=HF_TOKEN)
phase3_files = [f for f in repo_files if "phase_3_rr/" in f.lower()]
if not phase3_files:
    raise FileNotFoundError("No Phase_3_RR files found. Check access + token.")

def download_phase3_file(filename):
    target = filename.lower()
    matches = [f for f in phase3_files if Path(f).name.lower() == target]
    if not matches:
        matches = [f for f in phase3_files if normalized_name(Path(f)).lower() == target]
    if not matches:
        raise FileNotFoundError(f"Could not find {filename} under Phase_3_RR.")
    if "test" in target:
        preferred = [f for f in matches if "/test/" in f.lower()]
    elif "label_vocab" in target:
        preferred = matches
    else:
        preferred = [f for f in matches if "/train/" in f.lower()]
    repo_path = (preferred or matches)[0]
    local_path = hf_hub_download(repo_id=HF_REPO, filename=repo_path, repo_type="dataset", token=HF_TOKEN)
    print(f"Downloaded {filename}")
    return Path(local_path)

CL_TRAIN  = download_phase3_file("CL_train.json")
CL_DEV    = download_phase3_file("CL_dev.json")
IT_TRAIN  = download_phase3_file("IT_train.json")
IT_DEV    = download_phase3_file("IT_dev.json")
CL_TEST   = download_phase3_file("CL_test.json")
IT_TEST   = download_phase3_file("IT_test.json")
LABEL_VOCAB = download_phase3_file("label_vocab.json")
CKPT_PATH = CHECKPOINT_PATH

if str(TOKENIZER_PY.parent) not in sys.path:
    sys.path.insert(0, str(TOKENIZER_PY.parent))

print("\nAll assets located.")

Mounted at /content/drive
Searching Google Drive for Phase 1 assets...
Hugging Face read token (input hidden): ··········


CL_train.json:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

Downloaded CL_train.json


CL_dev.json:   0%|          | 0.00/336k [00:00<?, ?B/s]

Downloaded CL_dev.json


IT_train.json:   0%|          | 0.00/1.51M [00:00<?, ?B/s]

... (5 outputs left out)

IT_test.json:   0%|          | 0.00/194k [00:00<?, ?B/s]

Downloaded IT_test.json


label_vocab.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

Downloaded label_vocab.json

All assets located.


In [ ]:
# CELL 2 — IMPORTS & DEVICE

import json, math, random, zipfile
from pathlib import Path
from dataclasses import dataclass
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler

from peft import LoraConfig, get_peft_model, TaskType
from a1_tokenizer import BPETokenizer

OUTPUT_DIR = DRIVE_ROOT / "CS781_Phase3_LoRA_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
print(f"Output: {OUTPUT_DIR}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name} ({props.total_memory/1024**3:.1f} GiB)")

In [ ]:
# CELL 3 — HYPERPARAMETERS

@dataclass
class TrainCfg:
    # Encoder arch (must match Phase 1)
    vocab_size: int = 40_001
    d_model: int    = 768
    n_layers: int   = 14
    n_heads: int    = 12
    d_ff: int       = 2048
    max_seq_len: int = 512
    rope_theta: float = 10_000.0

    # Fine-tuning head
    num_labels: int = 13
    dropout: float  = 0.1
    cls_hidden: int = 512
    cls_dropout: float = 0.3
    context_window: int = 512

    # LoRA
    lora_r: int        = 16
    lora_alpha: int    = 32
    lora_dropout: float = 0.1
    lora_target_modules = ("q", "k", "v", "o")

    # Training (LoRA tolerates longer training at higher LR)
    epochs: int        = 15
    batch_size: int    = 8
    grad_accum: int    = 4
    lr: float          = 2e-4
    weight_decay: float = 0.01
    warmup_ratio: float = 0.06
    max_grad_norm: float = 1.0
    label_smoothing: float = 0.0   # off — fights class weights
    class_weight_cap: float = 3.0  # cap PrecedentOverruled's extreme weight

    early_stop_patience: int = 3
    seed: int = 42

    def __post_init__(self):
        self.head_dim = self.d_model // self.n_heads

CFG = TrainCfg()
print(f"LoRA: r={CFG.lora_r}, alpha={CFG.lora_alpha}, targets={CFG.lora_target_modules}")
print(f"Batch size: {CFG.batch_size}, LR: {CFG.lr}, Epochs: {CFG.epochs}")
print(f"Early stop patience: {CFG.early_stop_patience}")

LoRA: r=16, alpha=32, targets=('q', 'k', 'v', 'o')
Batch size: 8, LR: 0.0002, Epochs: 15
Early stop patience: 3


In [ ]:
# CELL 4 — SEED + TOKENIZER

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = True   # faster on dynamic shapes
        torch.backends.cudnn.deterministic = False

seed_everything(CFG.seed)

tokenizer = BPETokenizer.load(str(TOKENIZER_JSON))
BASE_VOCAB_SIZE = int(tokenizer.vocab_size)
MASK_ID = BASE_VOCAB_SIZE
SPECIAL_TOKEN_MAP = getattr(tokenizer, "special_tokens", {}) or {}
PAD_ID = int(SPECIAL_TOKEN_MAP.get("[PAD]", 0))

assert CFG.vocab_size == BASE_VOCAB_SIZE + 1
print(f"Tokenizer vocab: {BASE_VOCAB_SIZE}, PAD_ID: {PAD_ID}, MASK_ID: {MASK_ID}")

Tokenizer vocab: 40000, PAD_ID: 0, MASK_ID: 40000


In [ ]:
# CELL 5 — LOAD DATA + CAPPED CLASS WEIGHTS

with open(LABEL_VOCAB) as f:
    LABEL2ID = json.load(f)
ID2LABEL = {v: k for k, v in LABEL2ID.items()}
NUM_LABELS = len(LABEL2ID)
CFG.num_labels = NUM_LABELS
print(f"Labels ({NUM_LABELS}): {list(LABEL2ID.keys())}")

def load_train_data(path):
    with open(path) as f:
        data = json.load(f)
    return [(d["id"], d["text"], [LABEL2ID[l] for l in d["labels"]]) for d in data]

def load_test_data(path):
    with open(path) as f:
        data = json.load(f)
    return [(doc_id, sentences) for doc_id, sentences in data.items()]

cl_train_docs = load_train_data(CL_TRAIN)
it_train_docs = load_train_data(IT_TRAIN)
cl_dev_docs   = load_train_data(CL_DEV)
it_dev_docs   = load_train_data(IT_DEV)
train_docs = cl_train_docs + it_train_docs
dev_docs   = cl_dev_docs + it_dev_docs
test_docs  = load_test_data(CL_TEST) + load_test_data(IT_TEST)

print(f"Train: {len(train_docs)} docs, {sum(len(d[1]) for d in train_docs)} sentences")
print(f"Dev:   {len(dev_docs)} docs, {sum(len(d[1]) for d in dev_docs)} sentences")
print(f"Test:  {len(test_docs)} docs")

# Class weights with capping to avoid PrecedentOverruled dominating
all_labels = [l for _, _, labels in train_docs for l in labels]
label_counts = Counter(all_labels)
total = sum(label_counts.values())

class_weights = torch.zeros(NUM_LABELS)
for label_id in range(NUM_LABELS):
    freq = label_counts.get(label_id, 1) / total
    class_weights[label_id] = 1.0 / math.sqrt(freq)
class_weights = class_weights / class_weights.mean()
class_weights = torch.clamp(class_weights, max=CFG.class_weight_cap)
class_weights = class_weights.to(DEVICE)

print(f"\nClass weights (capped at {CFG.class_weight_cap}):")
for i in range(NUM_LABELS):
    print(f"  {ID2LABEL[i]:<28} count={label_counts.get(i, 0):>5} weight={class_weights[i]:.3f}")

Labels (13): ['Fact', 'Issue', 'ArgumentPetitioner', 'ArgumentRespondent', 'Statute', 'Dissent', 'PrecedentReliedUpon', 'PrecedentNotReliedUpon', 'PrecedentOverruled', 'RulingByLowerCourt', 'RatioOfTheDecision', 'RulingByPresentCourt', 'None']
Train: 80 docs, 16845 sentences
Dev:   10 docs, 2142 sentences
Test:  10 docs

Class weights (capped at 3.0):
  Fact                         count= 4098 weight=0.192
  Issue                        count=  107 weight=1.186
  ArgumentPetitioner           count= 1613 weight=0.305
  ArgumentRespondent           count= 1348 weight=0.334
  Statute                      count= 1099 weight=0.370
  Dissent                      count=  284 weight=0.728
  PrecedentReliedUpon          count= 1945 weight=0.278
  PrecedentNotReliedUpon       count=  569 weight=0.514
  PrecedentOverruled           count=    3 weight=3.000
  RulingByLowerCourt           count=  874 weight=0.415
  RatioOfTheDecision           count= 4301 weight=0.187
  RulingByPresentCourt        

In [ ]:
# CELL 6 — DATASET: Context-window sentence classification

class RhetoricalRoleDataset(Dataset):
    def __init__(self, docs, tokenizer, max_len=512, is_test=False):
        self.samples = []
        self.max_len = max_len
        self.is_test = is_test

        for doc in docs:
            if is_test:
                doc_id, sentences = doc
                labels = [0] * len(sentences)
            else:
                doc_id, sentences, labels = doc

            sent_tokens = [tokenizer.encode(s) for s in sentences]

            for i in range(len(sentences)):
                target_ids = sent_tokens[i]
                if len(target_ids) == 0:
                    # Use a single PAD token so we still produce a prediction
                    target_ids = [PAD_ID]

                target_ids = target_ids[:max_len - 10]
                budget = max_len - len(target_ids)
                before_ids, after_ids = [], []

                for j in range(i - 1, -1, -1):
                    stoks = sent_tokens[j]
                    if len(before_ids) + len(stoks) > budget // 2:
                        break
                    before_ids = stoks + before_ids

                remaining = budget - len(before_ids)
                for j in range(i + 1, len(sentences)):
                    stoks = sent_tokens[j]
                    if len(after_ids) + len(stoks) > remaining:
                        break
                    after_ids = after_ids + stoks

                input_ids = before_ids + target_ids + after_ids
                target_start = len(before_ids)
                target_end = target_start + len(target_ids)

                self.samples.append({
                    "input_ids": input_ids,
                    "target_start": target_start,
                    "target_end": target_end,
                    "label": labels[i],
                    "doc_id": doc_id,
                    "sent_idx": i,
                })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]


def collate_fn(batch):
    max_len = max(len(b["input_ids"]) for b in batch)
    input_ids, padding_mask = [], []
    target_starts, target_ends, labels = [], [], []
    for b in batch:
        ids = b["input_ids"]
        pad_len = max_len - len(ids)
        input_ids.append(ids + [PAD_ID] * pad_len)
        padding_mask.append([False] * len(ids) + [True] * pad_len)
        target_starts.append(b["target_start"])
        target_ends.append(b["target_end"])
        labels.append(b["label"])
    return {
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "padding_mask": torch.tensor(padding_mask, dtype=torch.bool),
        "target_starts": torch.tensor(target_starts, dtype=torch.long),
        "target_ends": torch.tensor(target_ends, dtype=torch.long),
        "labels": torch.tensor(labels, dtype=torch.long),
    }

train_ds   = RhetoricalRoleDataset(train_docs, tokenizer, max_len=CFG.context_window)
dev_ds     = RhetoricalRoleDataset(dev_docs,   tokenizer, max_len=CFG.context_window)
cl_dev_ds  = RhetoricalRoleDataset(cl_dev_docs, tokenizer, max_len=CFG.context_window)
it_dev_ds  = RhetoricalRoleDataset(it_dev_docs, tokenizer, max_len=CFG.context_window)
test_ds    = RhetoricalRoleDataset(test_docs,  tokenizer, max_len=CFG.context_window, is_test=True)

common = dict(collate_fn=collate_fn, num_workers=2, pin_memory=True, persistent_workers=True)
train_loader   = DataLoader(train_ds, batch_size=CFG.batch_size,       shuffle=True,  **common)
dev_loader     = DataLoader(dev_ds,   batch_size=CFG.batch_size * 2,   shuffle=False, **common)
cl_dev_loader  = DataLoader(cl_dev_ds, batch_size=CFG.batch_size * 2,  shuffle=False, **common)
it_dev_loader  = DataLoader(it_dev_ds, batch_size=CFG.batch_size * 2,  shuffle=False, **common)
test_loader    = DataLoader(test_ds,  batch_size=CFG.batch_size * 2,   shuffle=False, **common)

print(f"Train samples: {len(train_ds)}")
print(f"Dev samples:   {len(dev_ds)} (CL: {len(cl_dev_ds)}, IT: {len(it_dev_ds)})")
print(f"Test samples:  {len(test_ds)}")

Train samples: 16845
Dev samples:   2142 (CL: 1396, IT: 746)
Test samples:  2197


In [ ]:
# CELL 7 — MODEL ARCHITECTURE (matches Phase 1 encoder)

import torch.utils.checkpoint

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        rms = torch.rsqrt(x.float().pow(2).mean(-1, keepdim=True) + self.eps)
        return (x.float() * rms).to(x.dtype) * self.weight

def precompute_rope(dim, max_len, theta=10000.0, device=None):
    freqs = 1.0 / (theta ** (torch.arange(0, dim, 2, device=device).float() / dim))
    t = torch.arange(max_len, device=device).float()
    angles = torch.outer(t, freqs)
    return angles.cos(), angles.sin()

def apply_rope(x, cos, sin):
    T = x.size(2)
    c = cos[:T].unsqueeze(0).unsqueeze(0)
    s = sin[:T].unsqueeze(0).unsqueeze(0)
    hd = x.size(-1) // 2
    x1, x2 = x[..., :hd], x[..., hd:]
    rotated = torch.cat([-x2, x1], dim=-1)
    c_exp = torch.cat([c.expand(-1, -1, -1, hd), c.expand(-1, -1, -1, hd)], dim=-1)
    s_exp = torch.cat([s.expand(-1, -1, -1, hd), s.expand(-1, -1, -1, hd)], dim=-1)
    return x * c_exp + rotated * s_exp

class MultiHeadSelfAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.nh = cfg.n_heads; self.hd = cfg.head_dim; self.dm = cfg.d_model
        self.q = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.k = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.v = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.o = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.attn_drop  = nn.Dropout(cfg.dropout)
        self.resid_drop = nn.Dropout(cfg.dropout)
        self.scale = 1.0 / math.sqrt(cfg.head_dim)
    def forward(self, x, cos, sin, padding_mask=None):
        B, T, _ = x.shape
        q = self.q(x).view(B, T, self.nh, self.hd).transpose(1, 2)
        k = self.k(x).view(B, T, self.nh, self.hd).transpose(1, 2)
        v = self.v(x).view(B, T, self.nh, self.hd).transpose(1, 2)
        q = apply_rope(q, cos, sin); k = apply_rope(k, cos, sin)
        w = torch.matmul(q, k.transpose(-2, -1)) * self.scale
        if padding_mask is not None:
            w = w.masked_fill(padding_mask.unsqueeze(1).unsqueeze(2), float("-inf"))
        w = F.softmax(w, dim=-1, dtype=torch.float32).to(q.dtype)
        w = self.attn_drop(w)
        out = torch.matmul(w, v).transpose(1, 2).contiguous().view(B, T, self.dm)
        return self.resid_drop(self.o(out))

class SwiGLU(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.gate = nn.Linear(cfg.d_model, cfg.d_ff, bias=False)
        self.up   = nn.Linear(cfg.d_model, cfg.d_ff, bias=False)
        self.down = nn.Linear(cfg.d_ff, cfg.d_model, bias=False)
        self.drop = nn.Dropout(cfg.dropout)
    def forward(self, x):
        return self.drop(self.down(F.silu(self.gate(x)) * self.up(x)))

class EncoderBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm1 = RMSNorm(cfg.d_model); self.attn = MultiHeadSelfAttention(cfg)
        self.norm2 = RMSNorm(cfg.d_model); self.ffn  = SwiGLU(cfg)
    def forward(self, x, cos, sin, padding_mask=None):
        x = x + self.attn(self.norm1(x), cos, sin, padding_mask)
        x = x + self.ffn(self.norm2(x))
        return x

class LegalEncoderForClassification(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb  = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.emb_drop = nn.Dropout(cfg.dropout)
        self.layers   = nn.ModuleList([EncoderBlock(cfg) for _ in range(cfg.n_layers)])
        self.norm     = RMSNorm(cfg.d_model)

        cos, sin = precompute_rope(cfg.head_dim, cfg.max_seq_len, cfg.rope_theta)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)

        # MLM head kept for checkpoint load (frozen afterwards)
        self.mlm_dense = nn.Linear(cfg.d_model, cfg.d_model)
        self.mlm_norm  = RMSNorm(cfg.d_model)
        self.mlm_head  = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.mlm_head.weight = self.tok_emb.weight  # tied

        # Classification head (new, fully trainable)
        self.cls_head = nn.Sequential(
            nn.Linear(cfg.d_model, cfg.cls_hidden),
            nn.GELU(),
            nn.Dropout(cfg.cls_dropout),
            nn.Linear(cfg.cls_hidden, cfg.num_labels),
        )

    def forward(self, input_ids, padding_mask=None, target_starts=None, target_ends=None):
        x = self.emb_drop(self.tok_emb(input_ids))
        cos, sin = self.rope_cos, self.rope_sin
        for layer in self.layers:
            x = layer(x, cos, sin, padding_mask)
        hidden = self.norm(x)

        # Mean-pool target span
        B, T, D = hidden.shape
        pos = torch.arange(T, device=hidden.device).unsqueeze(0)
        target_mask = (pos >= target_starts.unsqueeze(1)) & (pos < target_ends.unsqueeze(1))
        tmf = target_mask.unsqueeze(-1).float()
        pooled = (hidden * tmf).sum(dim=1) / tmf.sum(dim=1).clamp(min=1)
        return self.cls_head(pooled)

print("Architecture defined ✓")

Architecture defined ✓


In [ ]:
# CELL 8 — LOAD PRETRAINED + APPLY LORA

model = LegalEncoderForClassification(CFG).to(DEVICE)

ckpt = torch.load(CKPT_PATH, map_location=DEVICE, weights_only=False)
print(f"Checkpoint: step={ckpt.get('step', '?')}, val_loss={ckpt.get('val_loss', '?')}")

pretrained_state = ckpt["model"]
mapped = {}
for k, v in pretrained_state.items():
    if k in ("drop.weight", "drop.bias"):
        mapped[k.replace("drop", "emb_drop")] = v
    else:
        mapped[k] = v

result = model.load_state_dict(mapped, strict=False)
assert all("cls_head" in k for k in result.missing_keys), result.missing_keys
print(f"Loaded pretrained encoder (cls_head is new).")

# Freeze everything — LoRA adds trainable adapters, cls_head gets unfrozen by modules_to_save
for p in model.parameters():
    p.requires_grad = False

lora_config = LoraConfig(
    r=CFG.lora_r,
    lora_alpha=CFG.lora_alpha,
    target_modules=list(CFG.lora_target_modules),  # matches attn.q/k/v/o by name suffix
    lora_dropout=CFG.lora_dropout,
    bias="none",
    modules_to_save=["cls_head"],                  # cls_head fully trainable
)

model = get_peft_model(model, lora_config)
model = model.to(DEVICE)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"\nTrainable: {trainable:,} ({trainable/total*100:.3f}% of {total:,})")
model.print_trainable_parameters()

Checkpoint: step=29000, val_loss=1.69698847756069
Loaded pretrained encoder (cls_head is new).

Trainable: 1,776,653 (1.340% of 132,601,882)
trainable params: 1,776,653 || all params: 132,601,882 || trainable%: 1.3398


In [ ]:
# CELL 9 — OPTIMIZER + SCHEDULER

trainable_params = [p for p in model.parameters() if p.requires_grad]

optimizer = torch.optim.AdamW(
    trainable_params,
    lr=CFG.lr,
    weight_decay=CFG.weight_decay,
)

total_steps = len(train_loader) * CFG.epochs // CFG.grad_accum
warmup_steps = max(1, int(total_steps * CFG.warmup_ratio))

def lr_lambda(step):
    if step < warmup_steps:
        return step / warmup_steps
    progress = (step - warmup_steps) / max(total_steps - warmup_steps, 1)
    return max(0.01, 0.5 * (1 + math.cos(math.pi * progress)))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
scaler = GradScaler("cuda")

print(f"Total steps: {total_steps}, warmup: {warmup_steps}, LR: {CFG.lr}")

Total steps: 7897, warmup: 473, LR: 0.0002


In [ ]:
# CELL 10 — TRAIN/EVAL FUNCTIONS

from sklearn.metrics import f1_score, classification_report

criterion = nn.CrossEntropyLoss(
    weight=class_weights,
    label_smoothing=CFG.label_smoothing,
)

def train_one_epoch(model, loader, optimizer, scheduler, scaler, epoch):
    model.train()
    total_loss = 0; total_correct = 0; total_samples = 0
    optimizer.zero_grad()

    for step, batch in enumerate(loader):
        input_ids     = batch["input_ids"].to(DEVICE)
        padding_mask  = batch["padding_mask"].to(DEVICE)
        target_starts = batch["target_starts"].to(DEVICE)
        target_ends   = batch["target_ends"].to(DEVICE)
        labels        = batch["labels"].to(DEVICE)

        with autocast(device_type="cuda", dtype=torch.float16):
            logits = model(
                input_ids=input_ids,
                padding_mask=padding_mask,
                target_starts=target_starts,
                target_ends=target_ends,
            )
            loss = criterion(logits, labels) / CFG.grad_accum

        scaler.scale(loss).backward()

        should_step = (step + 1) % CFG.grad_accum == 0 or (step + 1) == len(loader)
        if should_step:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(trainable_params, CFG.max_grad_norm)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()

        total_loss += loss.item() * CFG.grad_accum
        preds = logits.argmax(dim=-1)
        total_correct += (preds == labels).sum().item()
        total_samples += labels.size(0)

    return total_loss / max(len(loader), 1), total_correct / max(total_samples, 1)


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    total_loss = 0

    for batch in loader:
        input_ids     = batch["input_ids"].to(DEVICE)
        padding_mask  = batch["padding_mask"].to(DEVICE)
        target_starts = batch["target_starts"].to(DEVICE)
        target_ends   = batch["target_ends"].to(DEVICE)
        labels        = batch["labels"].to(DEVICE)

        with autocast(device_type="cuda", dtype=torch.float16):
            logits = model(
                input_ids=input_ids,
                padding_mask=padding_mask,
                target_starts=target_starts,
                target_ends=target_ends,
            )
            loss = criterion(logits, labels)

        total_loss += loss.item()
        preds = logits.argmax(dim=-1)
        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(labels.cpu().tolist())

    all_ids = list(range(NUM_LABELS))
    macro_f1 = f1_score(all_labels, all_preds, labels=all_ids, average="macro", zero_division=0)
    acc = sum(p == l for p, l in zip(all_preds, all_labels)) / max(len(all_preds), 1)
    return total_loss / max(len(loader), 1), macro_f1, acc, all_preds, all_labels

print("Train/eval functions defined ✓")

Train/eval functions defined ✓


In [ ]:
# CELL 11 — GPU PREFLIGHT

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU detected.")

model.train()
optimizer.zero_grad(set_to_none=True)
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

print("GPU:", torch.cuda.get_device_name(0))
print(f"Batch size: {CFG.batch_size}, LR: {CFG.lr}")

try:
    batch = next(iter(train_loader))
    input_ids     = batch["input_ids"].to(DEVICE)
    padding_mask  = batch["padding_mask"].to(DEVICE)
    target_starts = batch["target_starts"].to(DEVICE)
    target_ends   = batch["target_ends"].to(DEVICE)
    labels        = batch["labels"].to(DEVICE)

    print("Batch shape:", tuple(input_ids.shape))

    with autocast(device_type="cuda", dtype=torch.float16):
        logits = model(
            input_ids=input_ids,
            padding_mask=padding_mask,
            target_starts=target_starts,
            target_ends=target_ends,
        )
        loss = criterion(logits, labels) / CFG.grad_accum

    scaler.scale(loss).backward()
    torch.cuda.synchronize()

    peak_alloc = torch.cuda.max_memory_allocated() / 1024**3
    peak_res   = torch.cuda.max_memory_reserved() / 1024**3
    total      = torch.cuda.get_device_properties(0).total_memory / 1024**3

    print(f"\nFWD+BWD OK | Loss: {loss.item() * CFG.grad_accum:.4f}")
    print(f"Peak allocated: {peak_alloc:.2f} GiB | reserved: {peak_res:.2f} GiB | total: {total:.2f} GiB")
    print(f"Headroom: {total - peak_res:.2f} GiB")

except torch.cuda.OutOfMemoryError:
    print("\nOOM. Reduce CFG.batch_size and retry.")
    raise
finally:
    optimizer.zero_grad(set_to_none=True)
    torch.cuda.empty_cache()

GPU: NVIDIA L4
Batch size: 8, LR: 0.0002
Batch shape: (8, 511)

FWD+BWD OK | Loss: 2.5862
Peak allocated: 6.37 GiB | reserved: 6.77 GiB | total: 22.03 GiB
Headroom: 15.27 GiB


In [ ]:
# CELL 12 — TRAIN WITH EARLY STOPPING

import time

best_f1 = 0.0
best_epoch = -1
no_improve = 0
history = []

BEST_CKPT = OUTPUT_DIR / "best_lora.pt"

print("=" * 70)
print("LORA TRAINING")
print("=" * 70)
start_time = time.time()

for epoch in range(CFG.epochs):
    t0 = time.time()
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, scheduler, scaler, epoch)
    val_loss, val_f1, val_acc, _, _ = evaluate(model, dev_loader)
    epoch_time = time.time() - t0

    lr_now = optimizer.param_groups[0]["lr"]
    history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss, "train_acc": train_acc,
        "val_loss": val_loss, "val_f1": val_f1, "val_acc": val_acc,
    })

    improved = ""
    if val_f1 > best_f1:
        best_f1 = val_f1
        best_epoch = epoch + 1
        no_improve = 0
        torch.save({
            "epoch": epoch + 1,
            "model": model.state_dict(),
            "val_f1": val_f1,
            "val_loss": val_loss,
        }, BEST_CKPT)
        improved = " ★ saved"
    else:
        no_improve += 1

    print(f"Epoch {epoch+1:2d}/{CFG.epochs} │ "
          f"train_loss={train_loss:.4f} acc={train_acc:.3f} │ "
          f"val_loss={val_loss:.4f} F1={val_f1:.4f} acc={val_acc:.3f} │ "
          f"lr={lr_now:.2e} │ {epoch_time:.0f}s{improved}")

    if no_improve >= CFG.early_stop_patience:
        print(f"\nEarly stop: no improvement for {CFG.early_stop_patience} epochs")
        break

total_time = time.time() - start_time
print(f"\nTotal time: {total_time/60:.1f} min")
print(f"Best Macro-F1: {best_f1:.4f} at epoch {best_epoch}")

LORA TRAINING

Starting epoch 1 | 2106 batches
Epoch 1 | Batch 10/2106 (0.5%) | loss=2.5328 | avg_loss=2.5902 | acc=0.075 | speed=3.53 batches/s | elapsed=0.0 min | ETA=9.9 min | VRAM=2.0/6.8 GiB
Epoch 1 | Batch 20/2106 (0.9%) | loss=2.5665 | avg_loss=2.5826 | acc=0.069 | speed=3.67 batches/s | elapsed=0.1 min | ETA=9.5 min | VRAM=2.0/6.8 GiB
Epoch 1 | Batch 30/2106 (1.4%) | loss=2.5924 | avg_loss=2.5809 | acc=0.050 | speed=3.72 batches/s | elapsed=0.1 min | ETA=9.3 min | VRAM=2.0/6.8 GiB
Epoch 1 | Batch 40/2106 (1.9%) | loss=2.6179 | avg_loss=2.5762 | acc=0.066 | speed=3.76 batches/s | elapsed=0.2 min | ETA=9.2 min | VRAM=2.0/6.8 GiB
Epoch 1 | Batch 50/2106 (2.4%) | loss=2.5361 | avg_loss=2.5734 | acc=0.072 | speed=3.78 batches/s | elapsed=0.2 min | ETA=9.1 min | VRAM=2.0/6.8 GiB
Epoch 1 | Batch 60/2106 (2.8%) | loss=2.5965 | avg_loss=2.5746 | acc=0.069 | speed=3.79 batches/s | elapsed=0.3 min | ETA=9.0 min | VRAM=2.0/6.8 GiB
Epoch 1 | Batch 70/2106 (3.3%) | loss=2.5786 | avg_loss=2.5

In [ ]:
# CELL 13 — DETAILED DEV EVAL (CL + IT separately)

best_state = torch.load(BEST_CKPT, map_location=DEVICE, weights_only=False)
model.load_state_dict(best_state["model"])
print(f"Loaded best epoch {best_state['epoch']}, F1={best_state['val_f1']:.4f}\n")

target_names = [ID2LABEL[i] for i in range(NUM_LABELS)]
label_ids = list(range(NUM_LABELS))

# Combined dev
val_loss, val_f1, val_acc, val_preds, val_labels = evaluate(model, dev_loader)
print(f"COMBINED DEV: loss={val_loss:.4f}, Macro-F1={val_f1:.4f}, acc={val_acc:.4f}")
print(classification_report(val_labels, val_preds, labels=label_ids,
                             target_names=target_names, digits=4, zero_division=0))

# CL dev
cl_loss, cl_f1, cl_acc, _, _ = evaluate(model, cl_dev_loader)
print(f"\nCL DEV: Macro-F1={cl_f1:.4f}, acc={cl_acc:.4f}")

# IT dev
it_loss, it_f1, it_acc, _, _ = evaluate(model, it_dev_loader)
print(f"IT DEV: Macro-F1={it_f1:.4f}, acc={it_acc:.4f}")

Loaded best epoch 3, F1=0.4235

COMBINED DEV: loss=1.8877, Macro-F1=0.4235, acc=0.5215
                        precision    recall  f1-score   support

                  Fact     0.5232    0.6869    0.5940       444
                 Issue     0.7143    0.6667    0.6897        15
    ArgumentPetitioner     0.5741    0.4650    0.5138       400
    ArgumentRespondent     0.5909    0.3346    0.4272       272
               Statute     0.8148    0.7333    0.7719       120
               Dissent     0.0000    0.0000    0.0000         0
   PrecedentReliedUpon     0.3758    0.7087    0.4911       254
PrecedentNotReliedUpon     0.2692    0.2800    0.2745        50
    PrecedentOverruled     0.0000    0.0000    0.0000         0
    RulingByLowerCourt     0.7188    0.2255    0.3433       102
    RatioOfTheDecision     0.5701    0.4541    0.5055       403
  RulingByPresentCourt     0.6279    0.5510    0.5870        49
                  None     0.3125    0.3030    0.3077        33

              a

In [ ]:
# CELL 14 — TEST PREDICTIONS

model.eval()
test_predictions = {}

with torch.no_grad():
    sample_cursor = 0
    for batch in test_loader:
        input_ids     = batch["input_ids"].to(DEVICE)
        padding_mask  = batch["padding_mask"].to(DEVICE)
        target_starts = batch["target_starts"].to(DEVICE)
        target_ends   = batch["target_ends"].to(DEVICE)

        with autocast(device_type="cuda", dtype=torch.float16):
            logits = model(
                input_ids=input_ids,
                padding_mask=padding_mask,
                target_starts=target_starts,
                target_ends=target_ends,
            )
        preds = logits.argmax(dim=-1).cpu().tolist()

        for pred in preds:
            if sample_cursor >= len(test_ds.samples):
                raise ValueError("More predictions than samples.")
            sample = test_ds.samples[sample_cursor]
            key = (sample["doc_id"], sample["sent_idx"])
            if key in test_predictions:
                raise ValueError(f"Duplicate prediction for {key}")
            test_predictions[key] = ID2LABEL[pred]
            sample_cursor += 1

assert sample_cursor == len(test_ds.samples)

prediction = {}
for doc_id, sentences in test_docs:
    doc_labels = []
    for i in range(len(sentences)):
        key = (doc_id, i)
        if key not in test_predictions:
            raise ValueError(f"Missing prediction for {key}")
        doc_labels.append(test_predictions[key])
    prediction[doc_id] = doc_labels

assert len(prediction) == len(test_docs)
for doc_id, sentences in test_docs:
    assert len(prediction[doc_id]) == len(sentences)
    assert all(l in LABEL2ID for l in prediction[doc_id])

pred_path = OUTPUT_DIR / "prediction.json"
with open(pred_path, "w", encoding="utf-8") as f:
    json.dump(prediction, f, indent=2, ensure_ascii=False)

print(f"Saved: {pred_path}")
print(f"Documents: {len(prediction)}, total sentences: {sum(len(v) for v in prediction.values())}")

pred_counts = Counter(l for labels in prediction.values() for l in labels)
print("\nPrediction distribution:")
for label, count in pred_counts.most_common():
    print(f"  {label:<28} {count}")

Saved: /content/drive/MyDrive/CS781_Phase3_LoRA_outputs/prediction.json
Documents: 10, total sentences: 2197

Prediction distribution:
  PrecedentReliedUpon          625
  RatioOfTheDecision           443
  Fact                         419
  ArgumentPetitioner           273
  Statute                      183
  RulingByLowerCourt           101
  None                         46
  RulingByPresentCourt         37
  ArgumentRespondent           36
  PrecedentNotReliedUpon       17
  Issue                        15
  Dissent                      2


In [ ]:
# CELL 15 — SUBMISSION ZIP

zip_path = OUTPUT_DIR / "submission.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(pred_path, "prediction.json")

print(f"Created: {zip_path} ({zip_path.stat().st_size/1024:.1f} KB)")

with zipfile.ZipFile(zip_path, "r") as zf:
    print(f"Contents: {zf.namelist()}")
    with zf.open("prediction.json") as f:
        check = json.load(f)
    print(f"Docs: {len(check)}, sentences: {sum(len(v) for v in check.values())}")

Created: /content/drive/MyDrive/CS781_Phase3_LoRA_outputs/submission.zip (1.6 KB)
Contents: ['prediction.json']
Docs: 10, sentences: 2197


In [ ]:
# CELL 16 — TRAINING HISTORY

print(f"{'Epoch':>5} {'Train Loss':>11} {'Train Acc':>10} {'Val Loss':>9} {'Val F1':>8} {'Val Acc':>8}")
print("─" * 60)
for h in history:
    marker = " ★" if h["val_f1"] == best_f1 else ""
    print(f"  {h['epoch']:3d}   {h['train_loss']:>10.4f}  {h['train_acc']:>9.4f} "
          f" {h['val_loss']:>8.4f} {h['val_f1']:>7.4f} {h['val_acc']:>7.4f}{marker}")
print(f"\nBest: epoch {best_epoch}, Macro-F1 {best_f1:.4f}")

Epoch  Train Loss  Train Acc  Val Loss   Val F1  Val Acc
────────────────────────────────────────────────────────────
    1       1.7377     0.4597    1.4758  0.4141  0.5266
    2       0.7822     0.7447    1.7057  0.4146  0.5131
    3       0.4890     0.8369    1.8877  0.4235  0.5215 ★
    4       0.3741     0.8758    1.9487  0.4177  0.5266
    5       0.2924     0.9016    2.1750  0.4086  0.5182
    6       0.2353     0.9196    2.1465  0.4211  0.5626

Best: epoch 3, Macro-F1 0.4235
